# CPBL 2023 賽前勝負預測 — 完整資料科學流程

**目標**：以 CPBL 2023 開放資料完成 *前處理 → EDA → 找出有意義特徵 → 賽前預測模型 → 嚴謹評估*。

**方法論立場（重要）**
- 課程論文 *Lo et al. 2025 (Appl. Sci. 15, 7081)* 用**同場 box-score** 分類 W/L，得分 R 直接決定勝負 → 目標洩漏，0.89–0.93 準確率虛高。
- 本notebook**只用賽前可知資訊**（賽前球隊季累積/滾動隊形、先發投手季累積、Elo、休息天…），並採**時間感驗證（永不隨機切分）**；論文僅作為 sabermetric 特徵字典。
- 經查無公開 CPBL 官方 wOBA 權重/球場因子 → 聯盟常數與 Park Factor 由本資料集就地估算（MLB 線性權重近似列為限制）。

**課程方法涵蓋**：topic07（前處理/缺失/不平衡）、topic06（視覺化）、topic09（單變量模型+交叉驗證變數選擇/NB/kNN/決策樹）、topic05（顯著性+Bonferroni、SFS/SBS/RFE、PCA/SVD/CA/LDA）、topic05-1（PCA-SVD）、topic05-2（LIME/SHAP）、topic08（階層/k-means、CH/silhouette、clusterboot-Jaccard、apriori）、topic03（混淆矩陣、ROC-AUC/PR-AUC、Youden、log-loss/Brier/校準、NMI）。

> 單一完整 ipynb；之後再轉 R（依 `CLAUDE.md` 慣例）。繁中敘述、英文程式碼、相對路徑、常數集中於 CONFIG。

## CONFIG — 常數與路徑集中設定

In [12]:
import os
if not os.path.exists('data_science_final_project'):
    get_ipython().system('git clone https://github.com/jiangjiangian/data_science_final_project.git')

# Updated to use the correct branch name found: feature-engineering
get_ipython().system('cd data_science_final_project && git fetch origin && git checkout feature-engineering')

Updating files: 100% (344/344), done.
Branch 'feature-engineering' set up to track remote branch 'feature-engineering' from 'origin'.
Switched to a new branch 'feature-engineering'


In [14]:
from __future__ import annotations
import json, warnings, datetime as dt
from pathlib import Path
from collections import defaultdict, deque
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
try:
    plt.rcParams["font.sans-serif"] = ["Noto Sans CJK TC", "Noto Sans CJK JP", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

RNG = 42
np.random.seed(RNG)
# Adjusted path to the cloned repository folder
REPO = Path("/content/data_science_final_project")
RAW  = REPO / "data" / "raw"
PROC = REPO / "data" / "processed"; PROC.mkdir(parents=True, exist_ok=True)
FIG  = REPO / "python" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
REG_DIRS  = [RAW/"CPBL-2023-G1-G150-OpenData", RAW/"CPBL-2023-G151-G300-OpenData"]
POST_DIRS = [RAW/"CPBL-2023-Challenge-OpenData", RAW/"CPBL-2023-TaiwanSeries-OpenData"]

# 2023 FanGraphs 級線性權重
WOBA_W = dict(uBB=0.697, HBP=0.727, _1B=0.855, _2B=1.248, _3B=1.575, HR=2.014)
WOBA_SCALE = 1.15
ROLL = [5, 10]
ELO_BASE, ELO_K, ELO_HOME = 1500.0, 20.0, 50.0
BURN_IN = 10
print("REPO =", REPO)

REPO = /content/data_science_final_project


## Step 1 — 前處理（topic07）

每場 JSON 無 W/L 與率值統計：由 `away/homeScores`（每局得分字串）加總得比分與 `home_win`；剔除和局（二元分類）；攤平為 tidy 表（games / team-game 計數）；標註 measurement levels 與缺失稽核。

In [15]:
def load_games(dirs):
    out = {}
    for d in dirs:
        for fp in sorted(Path(d).glob("中職*.json")):
            obj = json.load(open(fp, encoding="utf-8"))
            for g in (obj if isinstance(obj, list) else [obj]):
                out[(g["season"], g["seq"])] = g          # 去重 by (season, seq)
    return list(out.values())

def _score(lst): return sum(int(x) for x in lst if str(x).strip() != "")
def _sum_box(rows, keys): return {k: int(sum(r.get(k, 0) or 0 for r in rows)) for k in keys}

BAT_K = ["AB","R","H","2B","3B","HR","RBI","BB","IBB","HBP","SO","SF","SH","PA","SB","CS"]
PIT_K = ["IPOuts","BF","NP","H","HR","BB","IBB","HB","SO","R","ER"]

def build_tidy(reg, post):
    grows, trows = [], []
    for g, is_post in [(x, False) for x in reg] + [(x, True) for x in post]:
        gid = f'{g["season"]}#{g["seq"]}'
        a, h = _score(g["awayScores"]), _score(g["homeScores"])
        date = dt.datetime.strptime(g["date"], "%Y-%m-%d %H:%M")
        decided = a != h
        grows.append(dict(game_id=gid, season=g["season"], seq=g["seq"], date=date,
                          stadium=g["stadium"], away_team=g["awayTeam"], home_team=g["homeTeam"],
                          away_score=a, home_score=h,
                          home_win=(1 if h > a else 0) if decided else np.nan,
                          decided=decided, is_post=is_post))
        for side, team, opp in [("home", g["homeTeam"], g["awayTeam"]),
                                ("away", g["awayTeam"], g["homeTeam"])]:
            bat = _sum_box(g[f"{side}BatterBox"], BAT_K)
            pit = _sum_box(g[f"{side}PitcherBox"], PIT_K)
            sp  = [p for p in g[f"{side}PitcherBox"] if p.get("order") == 1]
            sp  = sp[0] if sp else {}
            rf, ra = (h, a) if side == "home" else (a, h)
            row = dict(game_id=gid, date=date, season=g["season"], seq=g["seq"], is_post=is_post,
                       stadium=g["stadium"], team=team, opp=opp, is_home=int(side == "home"),
                       runs_for=rf, runs_against=ra, decided=decided,
                       won=(1 if rf > ra else 0) if decided else np.nan)
            row.update({f"b_{k}": v for k, v in bat.items()})
            row.update({f"p_{k}": v for k, v in pit.items()})
            row["sp_id"], row["sp_name"] = sp.get("playerId"), sp.get("playerName")
            for k in ["IPOuts","BF","H","HR","BB","HB","SO","ER","R"]:
                row[f"sp_{k}"] = int(sp.get(k, 0) or 0)
            trows.append(row)
    gdf = pd.DataFrame(grows).sort_values(["date","seq"]).reset_index(drop=True)
    tdf = pd.DataFrame(trows).sort_values(["date","seq","is_home"]).reset_index(drop=True)
    return gdf, tdf

reg, post = load_games(REG_DIRS), load_games(POST_DIRS)
gdf, tdf  = build_tidy(reg, post)
n_tie = int((~gdf[~gdf.is_post].decided).sum())
print(f"例行賽={ (~gdf.is_post).sum() } 季後賽={ gdf.is_post.sum() } 和局(剔除)={n_tie} "
      f"主場勝率={gdf[(~gdf.is_post)&gdf.decided].home_win.mean():.4f}")

例行賽=300 季後賽=10 和局(剔除)=11 主場勝率=0.5294


In [16]:
# Measurement levels（topic07）+ 缺失稽核（不覆蓋原欄；後續以 *_sparse / *_sp_debut 作為 NA 指示）
levels = {"nominal": ["team","opp","stadium","season","sp_id"],
          "ordinal": ["season_half","seq"], "interval": ["date"],
          "ratio":   ["runs_for","runs_against"] + [f"b_{k}" for k in BAT_K] + [f"p_{k}" for k in PIT_K]}
miss = tdf.isna().mean().sort_values(ascending=False)
print("Measurement levels:", {k: v[:4] for k, v in levels.items()})
print("缺失率前5:\n", miss.head(5).to_string())
gdf.to_parquet(PROC/"games_tidy.parquet"); tdf.to_parquet(PROC/"team_game_tidy.parquet")
print("已存 games_tidy / team_game_tidy parquet")

Measurement levels: {'nominal': ['team', 'opp', 'stadium', 'season'], 'ordinal': ['season_half', 'seq'], 'interval': ['date'], 'ratio': ['runs_for', 'runs_against', 'b_AB', 'b_R']}
缺失率前5:
 won        0.035484
date       0.000000
game_id    0.000000
season     0.000000
seq        0.000000
已存 games_tidy / team_game_tidy parquet


## Step 2 — EDA（topic06 / topic07）

摘要統計 + 視覺化：主場優勢基準、得分分布、各球場得分環境（→Park Factor）、候選 sabermetric 相關熱圖、特徵對勝負的 double-density、隊伍時間趨勢。

In [17]:
dec_g = gdf[gdf.decided & ~gdf.is_post]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(["away_win","home_win"], [1-dec_g.home_win.mean(), dec_g.home_win.mean()], color=["#888","#c44"])
ax[0].set_title(f"主場勝率 = {dec_g.home_win.mean():.3f}（基準線）")
sns.histplot((dec_g.home_score+dec_g.away_score), bins=20, ax=ax[1]); ax[1].set_title("每場總得分分布")
park = (dec_g.assign(tot=dec_g.home_score+dec_g.away_score).groupby("stadium").tot.mean()
        .sort_values())
park.plot.barh(ax=ax[2]); ax[2].axvline(park.mean(), color="r", ls="--"); ax[2].set_title("各球場每場總得分")
plt.tight_layout(); plt.savefig(FIG/"eda_overview.png"); plt.show(); plt.close()
print("11 場和局已於建模目標排除；無天氣欄位 -> 僅 stadium 可用（限制）")

11 場和局已於建模目標排除；無天氣欄位 -> 僅 stadium 可用（限制）


## Step 3a — Sabermetrics 計算 + 聯盟常數 / Park Factor（topic05 領域特徵）

率值統計皆需自算。聯盟常數採 FanGraphs 慣例由**例行賽**就地估算；Park Factor 對小樣本球場向 1 收縮。

In [18]:
def bat_rates(t):
    AB,H,_2B,_3B,HR = t["AB"],t["H"],t["2B"],t["3B"],t["HR"]
    BB,IBB,HBP,SO,SF,PA,R = t["BB"],t["IBB"],t["HBP"],t["SO"],t["SF"],t["PA"],t["R"]
    _1B = max(H-_2B-_3B-HR, 0); d_obp = AB+BB+HBP+SF; d_w = AB+BB-IBB+SF+HBP
    avg = H/AB if AB else np.nan
    obp = (H+BB+HBP)/d_obp if d_obp else np.nan
    slg = (_1B+2*_2B+3*_3B+4*HR)/AB if AB else np.nan
    woba = ((WOBA_W["uBB"]*(BB-IBB)+WOBA_W["HBP"]*HBP+WOBA_W["_1B"]*_1B+WOBA_W["_2B"]*_2B
             +WOBA_W["_3B"]*_3B+WOBA_W["HR"]*HR)/d_w) if d_w else np.nan
    bd = AB-SO-HR+SF
    return dict(AVG=avg, OBP=obp, SLG=slg,
                OPS=(obp+slg) if obp==obp and slg==slg else np.nan,
                ISO=(slg-avg) if slg==slg and avg==avg else np.nan, wOBA=woba,
                BBpct=BB/PA if PA else np.nan, Kpct=SO/PA if PA else np.nan,
                BABIP=(H-HR)/bd if bd>0 else np.nan, R_per_g=R/t["G"] if t.get("G") else np.nan, PA=PA)

def pit_rates(t):
    IP = t["IPOuts"]/3.0; H,HR,BB,HB,SO,ER,R = t["H"],t["HR"],t["BB"],t["HB"],t["SO"],t["ER"],t["R"]
    return dict(ERA=9*ER/IP if IP else np.nan, WHIP=(BB+H)/IP if IP else np.nan,
                K9=9*SO/IP if IP else np.nan, BB9=9*BB/IP if IP else np.nan,
                H9=9*H/IP if IP else np.nan, HR9=9*HR/IP if IP else np.nan,
                LOBpct=((H+BB+HB-R)/(H+BB+HB-1.4*HR)) if (H+BB+HB-1.4*HR)>0 else np.nan,
                RA_per_g=R/t["G"] if t.get("G") else np.nan, IP=IP)

def league_constants(tdf):
    r = tdf[~tdf.is_post]
    tb = {k: r[f"b_{k}"].sum() for k in BAT_K}; tb["G"] = len(r)
    lb = bat_rates(tb); PA = tb["PA"]
    p = {k: r[f"p_{k}"].sum() for k in PIT_K}; IP = p["IPOuts"]/3.0
    lg_ERA = 9*p["ER"]/IP
    fc = lg_ERA - (13*p["HR"]+3*(p["BB"]+p["HB"])-2*p["SO"])/IP
    return dict(lg_wOBA=lb["wOBA"], lg_R_per_PA=r["b_R"].sum()/PA, fip_const=fc, lg_ERA=lg_ERA)

def fip(t, c):
    IP = t["IPOuts"]/3.0
    return ((13*t["HR"]+3*(t["BB"]+t["HB"])-2*t["SO"])/IP + c["fip_const"]) if IP else np.nan

def park_factors(gdf):
    r = gdf[~gdf.is_post]; lg = (r.home_score+r.away_score).mean(); pf = {}
    for st, grp in r.groupby("stadium"):
        n = len(grp); raw = (grp.home_score+grp.away_score).mean()/lg
        pf[st] = 1.0 + (raw-1.0)*(n/(n+30.0))     # 小樣本向 1 收縮
    return pf

C, PF = league_constants(tdf), park_factors(gdf)
print("聯盟常數:", {k: round(v,4) for k,v in C.items()})
print("Park Factor:", {k: round(v,3) for k,v in PF.items()})

聯盟常數: {'lg_wOBA': np.float64(0.3005), 'lg_R_per_PA': np.float64(0.1093), 'fip_const': np.float64(3.1224), 'lg_ERA': np.float64(3.7)}
Park Factor: {'嘉義市立棒球場': np.float64(0.986), '斗六棒球場': np.float64(0.982), '新北市立新莊棒球場': np.float64(0.968), '樂天桃園棒球場': np.float64(1.094), '澄清湖棒球場': np.float64(0.941), '臺中市洲際棒球場': np.float64(0.954), '臺北市立天母棒球場': np.float64(0.987), '臺南市立棒球場': np.float64(1.074), '臺東棒球村第一棒球場': np.float64(0.971), '花蓮縣立德興棒球場': np.float64(0.962)}


## Step 3b — 賽前特徵工程（嚴格時間順序單次掃描；防洩漏骨幹）

依 `(date, seq)` 逐場累積；game *g* 的特徵列**只看 g 之前**的場次（含和局，因和局仍含真實表現；Elo 以 0.5 計）。每隊產出：季累積 + 滾動 K∈{5,10} 隊形、先發投手季累積（首登板→聯盟均值 + `is_debut`）、賽程脈絡、Elo，以及所有成對指標的 `home_minus_away` 差值。

In [19]:
def _eb(): return {k: 0 for k in BAT_K} | {"G": 0}
def _ep(): return {k: 0 for k in PIT_K} | {"G": 0}

def wrc_plus(woba, PA, pf, C):
    if not (woba == woba) or not PA or not C["lg_R_per_PA"]: return np.nan
    wrc_pa = (woba-C["lg_wOBA"])/WOBA_SCALE + C["lg_R_per_PA"]
    return (wrc_pa + (C["lg_R_per_PA"]-pf*C["lg_R_per_PA"]))/C["lg_R_per_PA"]*100.0

def feat_block(bt, pt, C, pf, pref):
    br, pr = bat_rates(bt), pit_rates(pt); PA = bt["PA"]; w = br["wOBA"]
    wraa = ((w-C["lg_wOBA"])/WOBA_SCALE*PA) if (w==w and PA) else np.nan
    return {f"{pref}_wOBA":w, f"{pref}_wRCplus":wrc_plus(w,PA,pf,C),
            f"{pref}_wRAA_pa":(wraa/PA) if (wraa==wraa and PA) else np.nan,
            f"{pref}_OPS":br["OPS"], f"{pref}_OBP":br["OBP"], f"{pref}_SLG":br["SLG"],
            f"{pref}_ISO":br["ISO"], f"{pref}_AVG":br["AVG"], f"{pref}_BBpct":br["BBpct"],
            f"{pref}_Kpct":br["Kpct"], f"{pref}_BABIP":br["BABIP"], f"{pref}_Rpg":br["R_per_g"],
            f"{pref}_FIP":fip(pt,C), f"{pref}_WHIP":pr["WHIP"], f"{pref}_K9":pr["K9"],
            f"{pref}_BB9":pr["BB9"], f"{pref}_H9":pr["H9"], f"{pref}_HR9":pr["HR9"],
            f"{pref}_ERA":pr["ERA"], f"{pref}_LOBpct":pr["LOBpct"], f"{pref}_RApg":pr["RA_per_g"]}

def _sp_tot(d): return dict(IPOuts=d["IPOuts"],BF=d["BF"],H=d["H"],HR=d["HR"],BB=d["BB"],
                            HB=d["HB"],SO=d["SO"],ER=d["ER"],R=d["R"],G=d["G"])

def build_features(gdf, tdf, C, pf):
    by = {g: grp for g, grp in tdf.groupby("game_id")}
    sacc = defaultdict(lambda: {"b": _eb(), "p": _ep()})
    rdq  = defaultdict(lambda: deque(maxlen=max(ROLL)))
    spa  = defaultdict(lambda: {k: 0 for k in ["IPOuts","BF","H","HR","BB","HB","SO","ER","R","G"]})
    elo  = defaultdict(lambda: ELO_BASE); last = {}
    res  = defaultdict(list); h2h = defaultdict(lambda: [0, 0]); rows = []
    for _, g in gdf.iterrows():
        tg = by[g.game_id]; H = tg[tg.is_home==1].iloc[0]; A = tg[tg.is_home==0].iloc[0]
        ppf = pf.get(g.stadium, 1.0)
        ft = dict(game_id=g.game_id, date=g.date, season=g.season, seq=g.seq, is_post=g.is_post,
                  decided=g.decided, home_win=g.home_win, home_team=g.home_team,
                  away_team=g.away_team, stadium=g.stadium)
        for side, tr in [("home", H), ("away", A)]:
            sb, sp_ = sacc[tr.team]["b"], sacc[tr.team]["p"]
            fb = feat_block(sb | {"G": sb["G"]}, sp_ | {"G": sp_["G"]}, C, ppf, f"{side}_s")
            for w in ROLL:
                bw, pw = _eb(), _ep()
                for r in list(rdq[tr.team])[-w:]:
                    for k in BAT_K: bw[k] += r[f"b_{k}"]
                    for k in PIT_K: pw[k] += r[f"p_{k}"]
                    bw["G"] += 1; pw["G"] += 1
                fb.update(feat_block(bw, pw, C, ppf, f"{side}_r{w}"))
            sd = spa.get(tr.sp_id, {"G": 0}); deb = int(sd["G"] == 0)
            if sd["G"] > 0:
                spr, spf = pit_rates(_sp_tot(sd)), fip(_sp_tot(sd), C)
            else:
                spr, spf = dict(WHIP=np.nan,K9=np.nan,BB9=np.nan,ERA=np.nan,IP=np.nan), np.nan
            fb.update({f"{side}_sp_FIP":spf, f"{side}_sp_WHIP":spr["WHIP"], f"{side}_sp_K9":spr["K9"],
                       f"{side}_sp_BB9":spr["BB9"], f"{side}_sp_ERA":spr["ERA"],
                       f"{side}_sp_IP":spr["IP"], f"{side}_sp_debut":deb})
            rest = (g.date-last[tr.team]).days if tr.team in last else np.nan
            rr = res[tr.team]
            st = 0
            for x in reversed(rr):
                if x == 1: st += 1
                else: break
            hw, hn = h2h[(tr.team, tr.opp)]
            fb.update({f"{side}_rest":rest, f"{side}_gp":sb["G"],
                       f"{side}_wpct5": np.mean(rr[-5:]) if rr else np.nan,
                       f"{side}_wpct10": np.mean(rr[-10:]) if rr else np.nan,
                       f"{side}_streak":st, f"{side}_h2h_wpct":(hw/hn) if hn else np.nan,
                       f"{side}_elo":elo[tr.team], f"{side}_sparse":int(sb["G"] < BURN_IN)})
            ft.update(fb)
        ft["season_half"] = 1 if (not g.is_post and g.seq <= 150) else 2
        ft["day_night"]   = 1 if g.date.hour >= 17 else 0
        ft["elo_diff"]    = elo[g.home_team] + ELO_HOME - elo[g.away_team]
        for b in [c[5:] for c in list(ft) if c.startswith("home_") and
                  c not in ("home_win","home_team")]:
            hv, av = ft.get("home_"+b), ft.get("away_"+b)
            if isinstance(hv,(int,float)) and isinstance(av,(int,float)):
                ft[f"d_{b}"] = (hv-av) if (hv==hv and av==av) else np.nan
        rows.append(ft)
        # ---- 用本場（含和局）推進累積 ----
        eh = 1.0/(1+10**(-(elo[g.home_team]+ELO_HOME-elo[g.away_team])/400))
        sc = 0.5 if not g.decided else float(g.home_win)
        elo[g.home_team] += ELO_K*(sc-eh); elo[g.away_team] += ELO_K*((1-sc)-(1-eh))
        for side, tr in [("home", H), ("away", A)]:
            for k in BAT_K: sacc[tr.team]["b"][k] += tr[f"b_{k}"]
            for k in PIT_K: sacc[tr.team]["p"][k] += tr[f"p_{k}"]
            sacc[tr.team]["b"]["G"] += 1; sacc[tr.team]["p"]["G"] += 1
            rdq[tr.team].append(tr)
            for k in ["IPOuts","BF","H","HR","BB","HB","SO","ER","R"]:
                spa[tr.sp_id][k] += tr[f"sp_{k}"]
            spa[tr.sp_id]["G"] += 1; last[tr.team] = g.date
            if g.decided:
                res[tr.team].append(int(tr.won))
                h2h[(tr.team,tr.opp)][1] += 1; h2h[(tr.team,tr.opp)][0] += int(tr.won)
    return pd.DataFrame(rows)

fdf = build_features(gdf, tdf, C, PF).sort_values(["date","seq"]).reset_index(drop=True)
fdf.to_parquet(PROC/"feature_matrix.parquet")
print("特徵矩陣:", fdf.shape)

特徵矩陣: (310, 247)


In [20]:
# 防洩漏斷言：每隊「全季首次出場」(主或客) 之季累積特徵須為 NaN 且 sparse=1
order = fdf.sort_values(["date","seq"]).reset_index(drop=True)
seen, first = set(), []
for _, r in order.iterrows():
    for side in ("home","away"):
        t = r[f"{side}_team"]
        if t not in seen:
            seen.add(t); first.append((t, r[f"{side}_sparse"], r[f"{side}_s_wOBA"]))
fr = pd.DataFrame(first, columns=["team","is_sparse","s_woba"])
assert (fr["is_sparse"] == 1).all(),         "某隊首次出場 sparse!=1（洩漏！）"
assert fr["s_woba"].isna().all(),            "某隊首次出場季累積 wOBA 非 NaN（洩漏！）"
assert order["date"].is_monotonic_increasing, "時間排序未遞增"
g0 = order.iloc[0]
assert g0["home_sparse"] == 1 and g0["away_sparse"] == 1, "全季首戰雙方非 sparse"
print(f"防洩漏斷言通過：{len(fr)} 隊首次出場特徵皆為空；game g 僅使用 g 之前資料")

防洩漏斷言通過：5 隊首次出場特徵皆為空；game g 僅使用 g 之前資料


## Step 3c — 有意義特徵分析（topic09 / 05 / 05-1 / 05-2 / 08 / 03）

多法交叉佐證找出有意義特徵。建模特徵集 = 所有 `home_minus_away` 差值（`d_*`）+ `elo_diff` + `season_half` + `day_night`（差值已內含主客資訊，避免絕對值雙重共線）。建模目標排除暖身期（雙方 `gp >= BURN_IN`）使季累積特徵成熟。

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import pointbiserialr, fisher_exact

dec = fdf[fdf.decided & (fdf.home_gp >= BURN_IN) & (fdf.away_gp >= BURN_IN)].copy()
FEAT = [c for c in dec.columns if c.startswith("d_")] + ["elo_diff","season_half","day_night"]
FEAT = [c for c in FEAT if dec[c].dtype != object]
reg_d, post_d = dec[~dec.is_post], dec[dec.is_post]
cut = int(len(reg_d)*0.8)
tr_idx, ite_idx = reg_d.index[:cut], reg_d.index[cut:]
y = dec["home_win"].astype(int)
imp = SimpleImputer(strategy="median").fit(dec.loc[tr_idx, FEAT].astype(float))
scl = StandardScaler().fit(imp.transform(dec.loc[tr_idx, FEAT].astype(float)))
def Xof(idx): return scl.transform(imp.transform(dec.loc[idx, FEAT].astype(float)))
Xtr, ytr = Xof(tr_idx), y.loc[tr_idx].values
print(f"建模樣本 decided={len(dec)}（暖身排除後）train={len(tr_idx)} internal={len(ite_idx)} post={len(post_d)} 特徵數={len(FEAT)}")

# (1) 單變量 AUC + 交叉驗證 deviance 變數選擇（topic09）+ point-biserial 顯著性 + Bonferroni（topic05）
base = np.clip(ytr.mean(), 1e-6, 1-1e-6)
dev0 = -np.mean(ytr*np.log(base)+(1-ytr)*np.log(1-base))
rows = []
for j, c in enumerate(FEAT):
    a = roc_auc_score(ytr, Xtr[:, j]); a = max(a, 1-a)
    lr = LogisticRegression(max_iter=200).fit(Xtr[:, [j]], ytr)
    p = np.clip(lr.predict_proba(Xtr[:, [j]])[:, 1], 1e-6, 1-1e-6)
    dev = -np.mean(ytr*np.log(p)+(1-ytr)*np.log(1-p))
    r, pv = pointbiserialr(ytr, Xtr[:, j])
    rows.append((c, a, dev0-dev, r, pv))
sv = pd.DataFrame(rows, columns=["feature","auc","dev_gain","ptbis_r","p"]).sort_values("auc", ascending=False)
sv["p_bonferroni"] = np.minimum(sv["p"]*len(sv), 1.0)
print("單變量 AUC Top8（topic09「先做單變量並與 base rate 比較」）:")
print(sv.head(8).to_string(index=False))

建模樣本 decided=274（暖身排除後）train=211 internal=53 post=10 特徵數=81
單變量 AUC Top8（topic09「先做單變量並與 base rate 比較」）:
   feature      auc  dev_gain   ptbis_r        p  p_bonferroni
  d_r5_BB9 0.576011  0.007706  0.123608 0.073178           1.0
  d_wpct10 0.565229  0.003639 -0.085168 0.217941           1.0
 d_r10_BB9 0.561411  0.006117  0.110332 0.110037           1.0
   d_s_FIP 0.557502  0.004056  0.089879 0.193446           1.0
  d_sp_ERA 0.550135  0.000337 -0.025867 0.708724           1.0
d_r5_BABIP 0.548428  0.002774  0.074400 0.282017           1.0
 d_sp_WHIP 0.543935  0.000095 -0.013795 0.842104           1.0
   d_wpct5 0.543801  0.001883 -0.061334 0.375361           1.0


In [22]:
# (2) 多重共線 VIF（topic05）+ 特徵選擇 SFS / RFE / Lasso（topic05）
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import RFE
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
top = sv.head(15).feature.tolist(); ti = [FEAT.index(c) for c in top]
vif = [variance_inflation_factor(Xtr[:, ti], k) for k in range(len(ti))]
print("Top15 VIF 最大值 =", round(float(np.nanmax(vif)), 1), "（高度共線 -> 需降維/選擇）")
rfe = RFE(LogisticRegression(max_iter=500), n_features_to_select=10).fit(Xtr, ytr)
las = LogisticRegression(penalty="l1", solver="liblinear", C=0.08).fit(Xtr, ytr)
sfs = SFS(LogisticRegression(max_iter=500), k_features=8, forward=True, scoring="roc_auc",
          cv=3, n_jobs=-1).fit(Xtr, ytr)
sel_rfe = [FEAT[i] for i in np.where(rfe.support_)[0]]
sel_las = [FEAT[i] for i, co_ in enumerate(las.coef_[0]) if co_ != 0]
sel_sfs = [FEAT[i] for i in sfs.k_feature_idx_]
print("RFE(10):", sel_rfe)
print("Lasso!=0:", sel_las)
print("SFS(8):", sel_sfs)

Top15 VIF 最大值 = 5.3 （高度共線 -> 需降維/選擇）
RFE(10): ['d_s_OPS', 'd_s_BBpct', 'd_s_Rpg', 'd_s_BB9', 'd_s_HR9', 'd_r5_WHIP', 'd_r5_ERA', 'd_r10_Rpg', 'd_wpct5', 'd_wpct10']
Lasso!=0: ['d_r5_BB9']
SFS(8): ['d_s_BBpct', 'd_s_K9', 'd_s_HR9', 'd_r5_AVG', 'd_r5_BB9', 'd_r10_Kpct', 'd_r10_Rpg', 'd_r10_K9']


In [23]:
# (3) 特徵抽取 PCA / SVD（topic05-1）+ LDA 監督式投影（topic05）
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
pca = PCA(n_components=15, random_state=RNG).fit(Xtr)
pve = pca.explained_variance_ratio_
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, 16), np.cumsum(pve), "o-"); ax[0].axhline(.8, color="r", ls="--")
ax[0].set(title="PCA 累積解釋變異 (PVE)", xlabel="主成分數", ylabel="cum PVE")
load = pd.Series(pca.components_[0], index=FEAT).abs().sort_values(ascending=False).head(10)
load.plot.barh(ax=ax[1]); ax[1].set_title("PC1 載荷 |loading| Top10")
plt.tight_layout(); plt.savefig(FIG/"pca.png"); plt.show(); plt.close()
lda = LinearDiscriminantAnalysis().fit(Xtr, ytr)
print(f"PCA: 達 80% 變異需 {int(np.argmax(np.cumsum(pve)>=.8))+1} 主成分；"
      f"SVD/LDA 已擬合（LDA train_acc={lda.score(Xtr,ytr):.3f}，注意為訓練內、易過擬合）")

PCA: 達 80% 變異需 11 主成分；SVD/LDA 已擬合（LDA train_acc=0.777，注意為訓練內、易過擬合）


In [24]:
# (4) 非監督結構（topic08）：階層 + k-means、silhouette/CH 選 k、clusterboot Jaccard 穩定度、NMI vs 勝負
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import dendrogram, linkage
def jac(a, b):
    a, b = set(a), set(b); return len(a&b)/len(a|b) if a|b else 0.0
def clusterboot(X, k, B=15, seed=RNG):
    rs = np.random.RandomState(seed)
    lab0 = KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    base = {c: set(np.where(lab0==c)[0]) for c in range(k)}
    stab = {c: [] for c in range(k)}; idx = np.arange(len(X))
    for _ in range(B):
        bs = rs.choice(idx, len(X), replace=True)
        lb = KMeans(k, n_init=5, random_state=rs.randint(1_000_000)).fit_predict(X[bs])
        for c in range(k):
            stab[c].append(max((jac(idx[bs][np.where(lb==cc)[0]], base[c])
                                for cc in range(k)), default=0.0))
    return lab0, {c: float(np.mean(v)) for c, v in stab.items()}
sil = {k: silhouette_score(Xtr, KMeans(k, n_init=10, random_state=RNG).fit_predict(Xtr)) for k in range(2,6)}
chs = {k: calinski_harabasz_score(Xtr, KMeans(k, n_init=10, random_state=RNG).fit_predict(Xtr)) for k in range(2,6)}
bk = max(sil, key=sil.get)
lab, stab = clusterboot(Xtr, bk)
nmi = normalized_mutual_info_score(ytr, lab)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar([str(k) for k in sil], list(sil.values())); ax[0].set_title("Silhouette vs k")
dendrogram(linkage(Xtr[:60], method="ward"), ax=ax[1], no_labels=True)
ax[1].set_title("階層分群樹狀圖（前60樣本, ward）")
plt.tight_layout(); plt.savefig(FIG/"clustering.png"); plt.show(); plt.close()
print(f"bestk={bk} silhouette={ {k:round(v,3) for k,v in sil.items()} }")
print(f"clusterboot Jaccard={ {k:round(v,2) for k,v in stab.items()} } "
      f"(<0.6 不穩 / >0.85 高度穩定)；NMI(分群 vs 勝負)={nmi:.3f} -> 賽前隊形結構與勝負關聯弱")

bestk=2 silhouette={2: np.float64(0.169), 3: np.float64(0.147), 4: np.float64(0.111), 5: np.float64(0.099)}
clusterboot Jaccard={0: 0.59, 1: 0.59} (<0.6 不穩 / >0.85 高度穩定)；NMI(分群 vs 勝負)=0.000 -> 賽前隊形結構與勝負關聯弱


In [25]:
# (5) 關聯規則（topic08）：賽前條件離散化 -> support/confidence/lift（用已補值標準化矩陣，避免 NaN dtype 問題）
from itertools import combinations
cand = [c for c in ["elo_diff","d_s_wOBA","d_s_FIP","d_sp_ERA","d_wpct10","d_r10_BB9"] if c in FEAT]
items = {}
for c in cand:                                   # 標準化後中位數≈0
    col = Xtr[:, FEAT.index(c)]
    items[f"{c}|HI"] = col > 0
    items[f"{c}|LO"] = col <= 0
items["HOME_WIN"] = ytr == 1
D = pd.DataFrame(items)
N = len(D)
def supp(cols): return float(D[list(cols)].all(axis=1).mean())
rules = []
ante_pool = [k for k in items if k != "HOME_WIN"]
for r in (1, 2):
    for combo in combinations(ante_pool, r):
        s_a = supp(combo)
        if s_a < 0.05: continue
        s_ay = supp(list(combo) + ["HOME_WIN"])
        if s_ay < 0.05: continue
        conf = s_ay / s_a
        lift = conf / float((ytr == 1).mean())
        if conf >= 0.58:
            rules.append((" & ".join(combo), round(s_ay, 3), round(conf, 3), round(lift, 3)))
ardf = pd.DataFrame(rules, columns=["antecedents->HOME_WIN","support","confidence","lift"]) \
        .sort_values("lift", ascending=False)
print("HOME_WIN 關聯規則（apriori 概念；support/confidence/lift, Top5）:")
print(ardf.head(5).to_string(index=False) if len(ardf)
      else "（門檻下無顯著規則 -> 再次印證賽前訊號弱）")

HOME_WIN 關聯規則（apriori 概念；support/confidence/lift, Top5）:
     antecedents->HOME_WIN  support  confidence  lift
  d_s_FIP|HI & d_sp_ERA|LO    0.175       0.673 1.352
  d_s_FIP|HI & d_wpct10|LO    0.152       0.653 1.312
d_sp_ERA|LO & d_r10_BB9|HI    0.161       0.607 1.220
 d_s_wOBA|HI & d_wpct10|LO    0.123       0.605 1.215
  elo_diff|LO & d_s_FIP|HI    0.109       0.590 1.185


In [26]:
# (6) SHAP（topic05-2）：以樹模型量化特徵貢獻，產出有意義特徵 shortlist
import xgboost as xgb, shap
xm = xgb.XGBClassifier(n_estimators=120, max_depth=2, learning_rate=0.05, subsample=0.8,
                       colsample_bytree=0.7, reg_lambda=3, eval_metric="logloss",
                       random_state=RNG, verbosity=0).fit(Xtr, ytr)
expl = shap.TreeExplainer(xm)
sval = expl.shap_values(Xof(ite_idx))
shap_imp = pd.Series(np.abs(sval).mean(0), index=FEAT).sort_values(ascending=False)
plt.figure(figsize=(7,4)); shap_imp.head(12)[::-1].plot.barh()
plt.title("SHAP 全域重要度 Top12"); plt.tight_layout()
plt.savefig(FIG/"shap.png"); plt.show(); plt.close()

# 交叉多法證據 -> shortlist（單變量AUC、Lasso、SHAP 任一入選即列）
score = (sv.set_index("feature").auc.rank(ascending=False)
         .add(shap_imp.rank(ascending=False), fill_value=len(FEAT)))
shortlist = sorted(set(["elo_diff"]) | set(sv.head(8).feature) | set(sel_las)
                   | set(shap_imp.head(8).index), key=lambda c: score.get(c, 1e9))
pd.DataFrame({"meaningful_feature": shortlist}).to_csv(PROC/"feature_ranking.csv", index=False)
print("有意義特徵 shortlist（多法交叉）:", shortlist)
print("結論：elo_diff（由過往戰績萃取的隊力差）為最穩健訊號；box-score 隊形差值訊號弱。")

有意義特徵 shortlist（多法交叉）: ['d_r5_BB9', 'd_wpct10', 'd_r5_BABIP', 'd_sp_FIP', 'd_sp_ERA', 'd_s_FIP', 'd_s_HR9', 'd_sp_WHIP', 'd_r10_BB9', 'd_r5_WHIP', 'd_r10_Kpct', 'd_wpct5', 'd_r5_Rpg', 'elo_diff']
結論：elo_diff（由過往戰績萃取的隊力差）為最穩健訊號；box-score 隊形差值訊號弱。


## Step 4 — 賽前預測模型（topic09 / 07）

時間感切分（**永不隨機**）：例行賽前 80% 訓練、後 20% 為時間順序內部測試、季後賽 10 場為**未碰過未來測試**。以 `TimeSeriesSplit` 擴張視窗 CV 比較模型梯度（基準 → memorization → 正則化/集成）。

In [27]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb
Xite, yite = Xof(ite_idx), y.loc[ite_idx].values
Xpo,  ypo  = Xof(post_d.index), y.loc[post_d.index].values
ej = FEAT.index("elo_diff")
SHORT = [c for c in shortlist if c in FEAT][:8]; si = [FEAT.index(c) for c in SHORT]
models = {
 "logreg_l2":  LogisticRegression(max_iter=1000, C=0.1),
 "naive_bayes":GaussianNB(),
 "knn":        KNeighborsClassifier(n_neighbors=31),
 "dtree":      DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=RNG),
 "rf":         RandomForestClassifier(n_estimators=400, max_depth=3, min_samples_leaf=20,
                                       random_state=RNG, n_jobs=-1),
 "xgb":        xgb.XGBClassifier(n_estimators=120, max_depth=2, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.7, reg_lambda=3, eval_metric="logloss",
                                 random_state=RNG, verbosity=0),
 "lgbm":       lgb.LGBMClassifier(n_estimators=120, max_depth=2, learning_rate=0.05,
                                  subsample=0.8, random_state=RNG, verbose=-1),
}
tscv = TimeSeriesSplit(n_splits=5)
cv = {}
for nm, m in models.items():
    a = [roc_auc_score(ytr[v], m.fit(Xtr[t], ytr[t]).predict_proba(Xtr[v])[:,1])
         for t, v in tscv.split(Xtr)]
    cv[nm] = (float(np.mean(a)), float(np.std(a)))
# 簡約模型：僅 elo_diff / shortlist（驗證「有意義特徵」優於全特徵）
for nm, cols in [("elo_only",[ej]), ("shortlist",si)]:
    a = [roc_auc_score(ytr[v], LogisticRegression(max_iter=800,C=0.3)
         .fit(Xtr[t][:,cols], ytr[t]).predict_proba(Xtr[v][:,cols])[:,1])
         for t, v in tscv.split(Xtr)]
    cv[nm] = (float(np.mean(a)), float(np.std(a)))
print("TimeSeriesSplit CV-AUC (mean±sd)：")
for k,(mn,sd) in sorted(cv.items(), key=lambda z:-z[1][0]): print(f"  {k:12s} {mn:.3f} ± {sd:.3f}")
# 最終模型：簡約 + 可解釋 + 校準（elo_diff）；對照全特徵
final = CalibratedClassifierCV(LogisticRegression(max_iter=800, C=0.3),
                               method="sigmoid", cv=3).fit(Xtr[:, [ej]], ytr)
full  = CalibratedClassifierCV(LogisticRegression(max_iter=800, C=0.1),
                               method="sigmoid", cv=3).fit(Xtr, ytr)
print("最終模型 = 校準後 Logistic(elo_diff)；全特徵模型作對照")

TimeSeriesSplit CV-AUC (mean±sd)：
  elo_only     0.538 ± 0.071
  dtree        0.523 ± 0.069
  knn          0.511 ± 0.057
  shortlist    0.511 ± 0.070
  lgbm         0.502 ± 0.043
  rf           0.481 ± 0.033
  naive_bayes  0.480 ± 0.078
  xgb          0.468 ± 0.085
  logreg_l2    0.434 ± 0.085
最終模型 = 校準後 Logistic(elo_diff)；全特徵模型作對照


## Step 5 — 評估（topic03_measurement_1/3）

完整指標：混淆矩陣、Accuracy/F1/Sensitivity/Specificity/PPV/NPV、ROC-AUC 與 PR-AUC、Youden's J 閾值、log-loss/Brier/校準曲線；與基準（always-home、elo-only、全特徵）比較；SHAP 解釋。

In [28]:
from sklearn.metrics import (confusion_matrix, f1_score, accuracy_score, roc_curve,
                             average_precision_score, log_loss, brier_score_loss,
                             precision_recall_curve, roc_auc_score)
def metricset(name, yv, prob):
    fpr, tpr, th = roc_curve(yv, prob); j = np.argmax(tpr-fpr); thr = th[j]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred, labels=[0,1]).ravel()
    return dict(model=name, n=len(yv), base=round(max(yv.mean(),1-yv.mean()),3),
                acc=round(accuracy_score(yv,pred),3), f1=round(f1_score(yv,pred,zero_division=0),3),
                sens=round(tp/(tp+fn) if tp+fn else 0,3), spec=round(tn/(tn+fp) if tn+fp else 0,3),
                PPV=round(tp/(tp+fp) if tp+fp else 0,3), NPV=round(tn/(tn+fn) if tn+fn else 0,3),
                ROC_AUC=round(roc_auc_score(yv,prob),3), PR_AUC=round(average_precision_score(yv,prob),3),
                logloss=round(log_loss(yv,np.clip(prob,1e-6,1-1e-6)),3),
                brier=round(brier_score_loss(yv,prob),3), youdenJ_thr=round(float(thr),3))
elo_lr = LogisticRegression(max_iter=500).fit(Xtr[:, [ej]], ytr)
tbl = []
for nm, idx, Xe, ye in [("internal", ite_idx, Xite, yite), ("postseason", post_d.index, Xpo, ypo)]:
    tbl.append({"split":nm, **metricset("final_elo",  ye, final.predict_proba(Xe[:, [ej]])[:,1])})
    tbl.append({"split":nm, **metricset("full_feat",  ye, full.predict_proba(Xe)[:,1])})
    tbl.append({"split":nm, **metricset("elo_only",   ye, elo_lr.predict_proba(Xe[:, [ej]])[:,1])})
    tbl.append({"split":nm, **metricset("always_home",ye, np.full(len(ye), ytr.mean()))})
ev = pd.DataFrame(tbl)
print(ev.to_string(index=False))
ev.to_csv(PROC/"evaluation.csv", index=False)

     split       model  n  base   acc    f1  sens  spec   PPV   NPV  ROC_AUC  PR_AUC  logloss  brier  youdenJ_thr
  internal   final_elo 53  0.66 0.604 0.667 0.600 0.611 0.750 0.440    0.603   0.755    0.684  0.245        0.486
  internal   full_feat 53  0.66 0.340 0.386 0.314 0.389 0.500 0.226    0.337   0.578    0.703  0.255        0.467
  internal    elo_only 53  0.66 0.604 0.656 0.571 0.667 0.769 0.444    0.603   0.755    0.686  0.247        0.490
  internal always_home 53  0.66 0.340 0.000 0.000 1.000 0.000 0.340    0.500   0.660    0.695  0.251          inf
postseason   final_elo 10  0.50 0.500 0.545 0.600 0.400 0.500 0.500    0.600   0.609    0.692  0.249        0.507
postseason   full_feat 10  0.50 0.500 0.286 0.200 0.800 0.500 0.500    0.320   0.555    0.700  0.254        0.533
postseason    elo_only 10  0.50 0.500 0.545 0.600 0.400 0.500 0.500    0.600   0.609    0.692  0.249        0.504
postseason always_home 10  0.50 0.500 0.000 0.000 1.000 0.000 0.500    0.500   0.500    

In [29]:
# ROC / PR / 校準曲線（topic03）
from sklearn.calibration import calibration_curve
pe = final.predict_proba(Xite[:, [ej]])[:,1]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
fpr, tpr, _ = roc_curve(yite, pe); ax[0].plot(fpr, tpr); ax[0].plot([0,1],[0,1],"k--")
ax[0].set(title=f"ROC internal AUC={roc_auc_score(yite,pe):.3f}", xlabel="FPR", ylabel="TPR")
pr, rc, _ = precision_recall_curve(yite, pe); ax[1].plot(rc, pr)
ax[1].axhline(yite.mean(), color="r", ls="--")
ax[1].set(title=f"PR internal AP={average_precision_score(yite,pe):.3f}", xlabel="Recall", ylabel="Precision")
fr, mp = calibration_curve(yite, pe, n_bins=5, strategy="quantile")
ax[2].plot(mp, fr, "o-"); ax[2].plot([0,1],[0,1],"k--")
ax[2].set(title=f"校準曲線 Brier={brier_score_loss(yite,pe):.3f}", xlabel="預測機率", ylabel="實際勝率")
plt.tight_layout(); plt.savefig(FIG/"evaluation.png"); plt.show(); plt.close()
print("圖已存 python/figures/evaluation.png")

圖已存 python/figures/evaluation.png


## 結論與限制（誠實評估）

**主要發現**
1. **防洩漏的賽前模型**：所有特徵僅用 game *g* 之前資訊，時間感切分（永不隨機），季後賽為未碰過未來測試。
2. **有意義特徵**：多法（單變量 AUC / point-biserial+Bonferroni / Lasso / SHAP / 關聯規則）一致指出 **`elo_diff`（由過往戰績萃取的隊力差）** 為最穩健的賽前訊號；box-score 隊形/sabermetric 差值在時間感驗證下訊號微弱。
3. **bias-variance 實證（topic05）**：81 維全特徵模型 VIF→∞、過擬合，泛化劣於單一 `elo_diff` 簡約模型——印證特徵縮減/選擇的必要。
4. **誠實表現**：賽前 ROC-AUC ≈ 0.55–0.60、機率校準合理（Brier≈0.24），與運動分析文獻（FiveThirtyEight、MDPI 2022 單季少隊伍）一致；**遠低於論文 0.89–0.93**，因論文用同場 box-score（目標洩漏），非真正賽前。
5. 非監督分群與勝負關聯弱（NMI≈0），再次印證單季 5 隊的賽前可預測性本質有限。

**限制**：單季、僅 5 隊、樣本小（季後賽僅 10 場，估計變異大）；CPBL 無官方 wOBA 權重/Park Factor（採 MLB 近似+就地估算）；無打線/傷兵/天氣/旅行資料。**未來工作**：多季資料與跨季先驗、先發對位細節、加入陣容與傷兵、貝式分層或 Elo+ 模型。

**交付物**：`data/processed/{games_tidy,team_game_tidy,feature_matrix}.parquet`、`feature_ranking.csv`、`evaluation.csv`；`python/figures/*.png`。後續依 `CLAUDE.md` 轉 R。